In [9]:
from langchain_openai.chat_models import ChatOpenAI
from langchain_core.prompts  import ChatPromptTemplate

chat = ChatOpenAI(
    temperature=0.0,
    model="qwen3:0.6b",
    base_url="http://localhost:11434/v1",
    api_key="ollama",
)

# other version: with ollama integration via langchain-ollama instead of langchain-opanai
# from langchain_ollama import ChatOllama
# chat = ChatOllama(
#     temperature=0.0,
#     model="qwen3:0.6b",
# )

In [10]:
my_template = """Extract the task name and its duration from the text below.
Return the response in {format} format, User Text: {text}
"""

prompt_template = ChatPromptTemplate.from_template(my_template)
user_text = "reading quran for 10 minutes each day"
final_prompt = prompt_template.format_messages(format="json", text=user_text)

print(final_prompt)

[HumanMessage(content='Extract the task name and its duration from the text below.\nReturn the response in json format, User Text: reading quran for 10 minutes each day\n', additional_kwargs={}, response_metadata={})]


In [13]:
llm_output = chat.invoke(final_prompt)

In [18]:
import json;
json.loads(llm_output.content)

{'task_name': 'reading Quran', 'duration': '10 minutes'}

In [23]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser

class ReviewExtraction(BaseModel):
    task_name: str = Field(
        description="Main task name. If the user mentions a sequence of tasks, choose one overall task name."
    )
    duration: int | None = Field(
        default=None,
        description="Duration in minutes if mentioned, otherwise null."
    )
    sub_tasks_chain: list[str] = Field(
        default_factory=list,
        description="List of sub-tasks if the user mentioned a sequence of tasks."
    )

output_parser = PydanticOutputParser(pydantic_object=ReviewExtraction)
format_instructions = output_parser.get_format_instructions()
print(format_instructions)


The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"task_name": {"description": "Main task name. If the user mentions a sequence of tasks, choose one overall task name.", "title": "Task Name", "type": "string"}, "duration": {"anyOf": [{"type": "integer"}, {"type": "null"}], "default": null, "description": "Duration in minutes if mentioned, otherwise null.", "title": "Duration"}, "sub_tasks_chain": {"description": "List of sub-tasks if the user mentioned a sequence of tasks.", "items": {"type": "string"}, "title": "Sub Tasks Chain", "type": "array"}}, "required": ["task_name"]}
